# Projet Data Science : Détection de Spams dans les Emails

## 1. Enoncé du Projet
Vous êtes chargé de développer une solution de détection automatique des spams à partir d'un ensemble de messages textuels. L'objectif est de construire un modèle d'apprentissage automatique capable de distinguer les messages 'spam' des messages légitimes ('ham'). Le projet doit également aboutir à une application web simple permettant à un utilisateur d'entrer un message et de recevoir une prédiction.

## 2. Objectifs Spécifiques
- Collecter et préparer les données.
- Appliquer un nettoyage adapté des textes.
- Extraire des caractéristiques pertinentes (TF-IDF).
- Entraîner et évaluer plusieurs modèles.
- Déployer une interface utilisateur Web simple.
- Rédiger une documentation claire du projet.

## 3. Étapes Recommandées
 - Étape 1 : Chargement et exploration des données
 - Étape 2 : Nettoyage et pré-traitement des textes
 - Étape 3 : Vectorisation des textes (TF-IDF)
 - Étape 4 : Séparation des données (train/test)
 - Étape 5 : Entraînement de plusieurs modèles (Naive Bayes, SVM, Random Forest)
 - Étape 6 : Évaluation via matrices de confusion, précision, rappel, F1-score
 - Étape 7 : Sélection du meilleur modèle
 - Étape 8 : Déploiement avec Streamlit

## 4. Recommandations
- Utiliser un jeu de données public fiable (ex : SMS Spam Collection).
- Documenter soigneusement toutes les étapes (Notebook ou README).
- Favoriser la simplicité pour le prototype initial, puis améliorer progressivement.
- Vérifier régulièrement l'équilibre entre les classes spam et ham.
- Privilégier un modèle interprétable au départ.
- Penser à l'optimisation des hyperparamètres (facultatif dans une 2ᵉ phase).
- S'assurer que l'application est intuitive et fonctionnelle.
---------------------------------------------------------------------------------------------------------

In [18]:
import pandas as pd
import re
import string
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

In [6]:
# Nettoyage du texte
def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{string.punctuation}]", "", text)
    text = re.sub(r"\d+", "", text)
    return text

In [8]:
# Charger données
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
data = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

In [9]:
data

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [10]:
data['label_num'] = data.label.map({'ham': 0, 'spam': 1})
data['message_clean'] = data['message'].apply(clean_text)

In [11]:
data

,label,message,label_num,message_clean
0,ham,"Go until jurong point, crazy.. Available only ...",0,go until jurong point crazy available only in ...
1,ham,Ok lar... Joking wif u oni...,0,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1,free entry in a wkly comp to win fa cup final...
3,ham,U dun say so early hor... U c already then say...,0,u dun say so early hor u c already then say
4,ham,"Nah I don't think he goes to usf, he lives aro...",0,nah i dont think he goes to usf he lives aroun...
...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,1,this is the nd time we have tried contact u u...
5568,ham,Will ü b going to esplanade fr home?,0,will ü b going to esplanade fr home
5569,ham,"Pity, * was in mood for that. So...any other s...",0,pity was in mood for that soany other suggest...
5570,ham,The guy did some bitching but I acted like i'd...,0,the guy did some bitching but i acted like id ...


In [12]:
# TF-IDF
tfidf = TfidfVectorizer(stop_words='english')
X = tfidf.fit_transform(data['message_clean'])
y = data['label_num']

In [13]:
# Séparation des données
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Entraînement du modèle Naive Bayes

In [14]:
model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [15]:
# Prédictions
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.9650224215246637
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       966
           1       1.00      0.74      0.85       149

    accuracy                           0.97      1115
   macro avg       0.98      0.87      0.91      1115
weighted avg       0.97      0.97      0.96      1115



## Avec SVM (Support Vector Machine)

In [17]:
model = LinearSVC()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy (SVM):", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy (SVM): 0.9838565022421525
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       966
           1       1.00      0.88      0.94       149

    accuracy                           0.98      1115
   macro avg       0.99      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115



## Avec Random Forest

In [19]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy (Random Forest):", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy (Random Forest): 0.9730941704035875
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       966
           1       1.00      0.80      0.89       149

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115



## 📊 Résumé des scores
| Modèle            | Accuracy | Précision (spam) | Rappel (spam) | F1-score (spam) |
|-------------------|----------|------------------|----------------|-----------------|
| Naive Bayes       | 0.965    | 1.00             | 0.74           | 0.85            |
| SVM (LinearSVC)   | **0.984**| 1.00             | **0.88**       | **0.94**        |
| Random Forest     | 0.973    | 1.00             | 0.80           | 0.89            |

## 🧠 Analyse
- Naive Bayes est rapide et donne de bons résultats, mais le rappel sur les spams est faible (0.74). Cela signifie qu’il manque plusieurs spams, ce qui est dangereux dans un système de détection.

- Random Forest améliore légèrement les performances, mais reste en dessous du modèle SVM.

- SVM (LinearSVC) a la meilleure accuracy générale (98.4%) et surtout le meilleur compromis précision / rappel / f1-score sur les spams, ce qui en fait le meilleur choix pour ce type de tâche.

## 💾 Code pour sauvegarder le modèle SVM

In [22]:
# Sauvegarde du modèle SVM
pickle.dump(model, open("model_svm.pkl", "wb"))

# Sauvegarde du vecteur TF-IDF
pickle.dump(tfidf, open("tfidf.pkl", "wb"))